<a href="https://colab.research.google.com/github/tauri-42/chain-of-thought-ode-reasoning/blob/main/transformer_feedforward_attribution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
!pip install -q -U inseq
!pip install -q "transformers==4.24.0"
!pip install sacremoses

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.0/110.0 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 314.9/314.9 kB 11.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 59.8 MB/s eta 0:00:00
  error: subprocess-exited-with-error
  
  × Building wheel for tokenizers (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for tokenizers
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (tokenizers)


In [4]:

import inseq

model = inseq.load_model(
    "Helsinki-NLP/opus-mt-en-it",
    "value_zeroing",
    model_kwargs={"attn_implementation": "eager"},
)

In [5]:
out = model.attribute(
    input_texts="Hello everyone, hope you're enjoying the tutorial!",
    output_step_attributions=True,
    show_progress=True,
)
out.show()

Attributing with value_zeroing...: 100%|██████████| 1/1 [00:00<?, ?it/s]

ValueError: Could not find assignment to value_states in MarianAttention(
  (k_proj): Linear(in_features=512, out_features=512, bias=True)
  (v_proj): Linear(in_features=512, out_features=512, bias=True)
  (q_proj): Linear(in_features=512, out_features=512, bias=True)
  (out_proj): Linear(in_features=512, out_features=512, bias=True)
)'s forward() method

In [14]:
import numpy as np

seq_attr = out.sequence_attributions[0]

source_attr = seq_attr.source_attributions.detach().cpu().numpy()

target_attr = None
if seq_attr.target_attributions is not None:
    target_attr = seq_attr.target_attributions.detach().cpu().numpy()

source_tokens = [t.token for t in seq_attr.source]
target_tokens = [t.token for t in seq_attr.target]

print("source_attr shape:", source_attr.shape)

source_attr shape: (14, 19, 512)


In [15]:
import h5py

def save_attribution(h5_path, example_id, source_attr, target_attr,
                      source_tokens, target_tokens, mode="a"):
    with h5py.File(h5_path, mode) as f:
        grp = f.create_group(f"example_{example_id}")
        grp.create_dataset("source_attr", data=source_attr, compression="gzip")
        if target_attr is not None:
            grp.create_dataset("target_attr", data=target_attr, compression="gzip")
        dt = h5py.string_dtype(encoding="utf-8")
        grp.create_dataset("source_tokens", data=np.array(source_tokens, dtype=object), dtype=dt)
        grp.create_dataset("target_tokens", data=np.array(target_tokens, dtype=object), dtype=dt)

save_attribution("attributions.h5", 0, source_attr, target_attr, source_tokens, target_tokens, mode="w")

Attributing with value_zeroing...: 100%|██████████| 1/1 [01:27<?, ?it/s]


In [16]:
texts = [
    "Hello everyone, hope you're enjoying the tutorial!",
    "This is a second example sentence.",
]

for i, text in enumerate(texts):
    out = model.attribute(input_texts=text, show_progress=False)
    seq_attr = out.sequence_attributions[0]

    source_attr = seq_attr.source_attributions.detach().cpu().numpy()
    target_attr = (seq_attr.target_attributions.detach().cpu().numpy()
                   if seq_attr.target_attributions is not None else None)
    source_tokens = [t.token for t in seq_attr.source]
    target_tokens = [t.token for t in seq_attr.target]

    save_attribution("attributions.h5", i, source_attr, target_attr,
                      source_tokens, target_tokens, mode="a")

    del out, seq_attr  # avoid accumulating in memory

TypeError: unsupported operand type(s) for +: 'Tensor' and 'tuple'

In [17]:
with h5py.File("attributions.h5", "r") as f:
    ex0 = f["example_0"]
    layer_3_scores = ex0["source_attr"][:, :, 3]
    tokens = [t.decode() if isinstance(t, bytes) else t for t in ex0["source_tokens"][:]]

print(layer_3_scores.shape, tokens)

(14, 19) ['▁H', 'ello', '▁everyone', ',', '▁hope', '▁you', "'", 're', '▁enjoying', '▁the', '▁tutor', 'ial', '!', '</s>']
